In [ ]:
# colab
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize
import warnings

warnings.filterwarnings('ignore')

print("1. Loading and Preprocessing Data...")
train = pd.read_csv('data/raw/Coderush-26-ML-Train.csv')
test = pd.read_csv('data/raw/Coderush-26-ML-test.csv')

label_map = {'lower': 0, 'middle': 1, 'upper': 2}
train['label'] = train['label'].map(label_map)

train['is_train'] = 1
test['is_train'] = 0
test['label'] = -1
df = pd.concat([train, test], ignore_index=True)

# --- Feature Engineering ---
df['age'] = 1994 - df['year_of_birth']
df['net_capital'] = df['capital_gain'] - df['capital_loss']
df['wealth_velocity'] = df['net_capital_asset'] / (df['age'] + 1)
df['financial_stress'] = df['poverty_line_usd'] / (df['hours_per_week'] + 1)
df['effort_yield'] = df['education_num'] * df['hours_per_week']
df['is_capital_active'] = ((df['capital_gain'] > 0) | (df['capital_loss'] > 0)).astype(int)

# Categorical Prep
df['education_tier'] = df['education_tier'].map({'Primary': 0, 'Secondary': 1, 'Higher': 2})
edu_map = {'Preschool': 0, '1st-4th': 1, '5th-6th': 2, '7th-8th': 3, '9th': 4, '10th': 5, '11th': 6, '12th': 7, 'HS-grad': 8, 'Some-college': 9, 'Assoc-voc': 10, 'Assoc-acdm': 11, 'Bachelors': 12, 'Masters': 13, 'Prof-school': 14, 'Doctorate': 15}
df['education'] = df['education'].map(edu_map)
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})

# Frequency Encoding
for col in ['native_country', 'occupation']:
    freq = df[col].value_counts() / len(df)
    df[f'{col}_freq'] = df[col].map(freq)

# Standard One-Hot
df_dummies = pd.get_dummies(df, columns=['relationship', 'race', 'workclass', 'marital_status', 'interview_mode', 'currency_code'], drop_first=True)
for col in df_dummies.columns:
    if df_dummies[col].dtype == 'bool': df_dummies[col] = df_dummies[col].astype(int)

X_train_full = df_dummies[df_dummies['is_train'] == 1].drop(['is_train', 'year_of_birth'], axis=1).copy()
X_test_full = df_dummies[df_dummies['is_train'] == 0].drop(['is_train', 'label', 'year_of_birth'], axis=1).copy()

# --- STAGE 1: Dual Model Weak Learners ---
print("\n2. STAGE 1: Training Ensemble of Weak Learners (LGBM + XGB)...")
features = [c for c in X_train_full.columns if c not in ['label', 'bag_id', 'occupation', 'native_country']]
X = X_train_full[features]
y = X_train_full['label']
groups = X_train_full['bag_id']

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((len(X_train_full), 3))
test_probs = np.zeros((len(X_test_full), 3))

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    lgb_m = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, num_leaves=31, importance_type='gain', random_state=fold, verbose=-1)
    # Fixed: Moved early_stopping_rounds to constructor
    xgb_m = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, tree_method='hist', random_state=fold, early_stopping_rounds=50)

    lgb_m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    fold_probs = (lgb_m.predict_proba(X_va) + xgb_m.predict_proba(X_va)) / 2
    oof_probs[val_idx] = fold_probs
    test_probs += ((lgb_m.predict_proba(X_test_full[features]) + xgb_m.predict_proba(X_test_full[features])) / 2) / 5

for i in range(3):
    X_train_full[f'prob_{i}'] = oof_probs[:, i]
    X_test_full[f'prob_{i}'] = test_probs[:, i]

# --- STAGE 2: Meta-Aggregation ---
print("\n3. STAGE 2: Aggregating Advanced Topology...")
def q25(x): return x.quantile(0.25)
def q75(x): return x.quantile(0.75)

agg_funcs = {}
for p in ['prob_0', 'prob_1', 'prob_2']: agg_funcs[p] = ['mean', 'max', 'std', q25, q75]
for f in ['wealth_velocity', 'education_num', 'age']: agg_funcs[f] = ['mean', 'max', 'min']

def get_meta(data, is_test=False):
    meta = data.groupby('bag_id').agg(agg_funcs)
    meta.columns = [f"{c[0]}_{c[1]}" for c in meta.columns]
    meta['bag_diversity'] = data.groupby('bag_id')['education'].nunique()
    if not is_test: meta['label'] = data.groupby('bag_id')['label'].first()
    return meta.fillna(0)

train_meta = get_meta(X_train_full)
test_meta = get_meta(X_test_full, is_test=True)

X_meta = train_meta.drop('label', axis=1)
y_meta = train_meta['label'].astype(int)

# Meta Training
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_meta = np.zeros((len(X_meta), 3))
final_test_probs = np.zeros((len(test_meta), 3))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_meta, y_meta)):
    m = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.01, max_depth=4, reg_lambda=5, random_state=42, verbose=-1)
    m.fit(X_meta.iloc[tr_idx], y_meta.iloc[tr_idx], eval_set=[(X_meta.iloc[va_idx], y_meta.iloc[va_idx])], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_meta[va_idx] = m.predict_proba(X_meta.iloc[va_idx])
    final_test_probs += m.predict_proba(test_meta) / 5

# --- Optimization ---
def opt(w): return -f1_score(y_meta, np.argmax(oof_meta * w, axis=1), average='macro')
res = minimize(opt, [1.0, 1.0, 1.1], method='Nelder-Mead')

print(f"\n🏆 IMPROVED MACRO F1: {-res.fun:.5f} 🏆")
final_preds = np.argmax(final_test_probs * res.x, axis=1)
submission = pd.DataFrame({'bag_id': test_meta.index, 'label': final_preds}).sort_values('bag_id')
submission.to_csv('submission_v5_improved.csv', index=False)
print("Saved to 'submission_v5_improved.csv'")

1. Loading and Preprocessing Data...

2. STAGE 1: Training Ensemble of Weak Learners (LGBM + XGB)...

3. STAGE 2: Aggregating Advanced Topology...

🏆 IMPROVED MACRO F1: 0.74132 🏆
Saved to 'submission_v5_improved.csv'


In [ ]:
#anas bhai
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize
import warnings

warnings.filterwarnings('ignore')

print("1. Loading and Preprocessing Data...")
train = pd.read_csv('data/raw/Coderush-26-ML-Train.csv')
test = pd.read_csv('data/raw/Coderush-26-ML-test.csv')

# Map labels
label_map = {'lower': 0, 'middle': 1, 'upper': 2}
train['label'] = train['label'].map(label_map)

# Combine for uniform processing
train['is_train'] = 1
test['is_train'] = 0
test['label'] = -1
df = pd.concat([train, test], ignore_index=True)

# =====================================================================
# ADVANCED INSTANCE-LEVEL FEATURE ENGINEERING
# =====================================================================
# 1. Base Derivations
df['age'] = 1994 - df['year_of_birth']
df['net_capital'] = df['capital_gain'] - df['capital_loss']

# 2. Financial & Effort Cross-Features (Crucial for Tree Splits)
df['wealth_velocity'] = df['net_capital_asset'] / (df['age'] + 1)
df['financial_stress'] = df['poverty_line_usd'] / (df['hours_per_week'] + 1)
df['effort_yield'] = df['education_num'] * df['hours_per_week']
df['is_capital_active'] = (df['capital_gain'] > 0) | (df['capital_loss'] > 0)
df['is_capital_active'] = df['is_capital_active'].astype(int)

# 3. Native Pandas Encoding (Zero Dependency on external libraries)
cols_to_drop = ['interview_mode', 'currency_code', 'year_of_birth']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Ordinal mappings
df['education_tier'] = df['education_tier'].map({'Primary': 0, 'Secondary': 1, 'Higher': 2})
edu_map = {
    'Preschool': 0, '1st-4th': 1, '5th-6th': 2, '7th-8th': 3, '9th': 4, '10th': 5,
    '11th': 6, '12th': 7, 'HS-grad': 8, 'Some-college': 9, 'Assoc-voc': 10,
    'Assoc-acdm': 11, 'Bachelors': 12, 'Masters': 13, 'Prof-school': 14, 'Doctorate': 15
}
df['education'] = df['education'].map(edu_map)
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})

# Frequency Encoding for high cardinality to avoid explosion of columns
for col in ['native_country', 'occupation']:
    freq = df[col].value_counts() / len(df)
    df[f'{col}_freq'] = df[col].map(freq)
    df = df.drop(col, axis=1)

# Standard One-Hot for the rest
df = pd.get_dummies(df, columns=['relationship', 'race', 'workclass', 'marital_status'], drop_first=True)
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

# Split back to Train / Test
X_train_full = df[df['is_train'] == 1].drop('is_train', axis=1).copy()
X_test_full = df[df['is_train'] == 0].drop(['is_train', 'label'], axis=1).copy()

# =====================================================================
# STAGE 1: INSTANCE-LEVEL WEAK LEARNER (Level 0)
# =====================================================================
print("\n2. STAGE 1: Training Instance Level Model (Weak Learners)...")
features = [c for c in X_train_full.columns if c not in ['label', 'bag_id']]
X = X_train_full[features]
y = X_train_full['label']
groups = X_train_full['bag_id']

# GroupKFold ensures no individuals from the same bag leak across train/val
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_instance_probs = np.zeros((len(X_train_full), 3))
test_instance_probs = np.zeros((len(X_test_full), 3))

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # Powerful, deep tree to extract maximum signal from individuals
    model_l0 = lgb.LGBMClassifier(
        n_estimators=1500, learning_rate=0.03, max_depth=7, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, class_weight='balanced', random_state=42, n_jobs=-1
    )
    model_l0.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])

    oof_instance_probs[val_idx] = model_l0.predict_proba(X_va)
    test_instance_probs += model_l0.predict_proba(X_test_full[features]) / 5

# Attach weak probabilities to original dataframes
for i in range(3):
    X_train_full[f'prob_{i}'] = oof_instance_probs[:, i]
    X_test_full[f'prob_{i}'] = test_instance_probs[:, i]

# =====================================================================
# STAGE 2: META-MODEL AGGREGATION & TRAINING (Level 1)
# =====================================================================
print("\n3. STAGE 2: Aggregating Topology & Training Meta-Model...")

def q25(x): return x.quantile(0.25)
def q75(x): return x.quantile(0.75)
def skew(x): return x.skew() if len(x) > 2 else 0

# The Ultimate Statistical Mapping Dictionary
agg_funcs = {
    'prob_0': ['mean', 'max', 'min', 'std', q25, q75, skew],
    'prob_1': ['mean', 'max', 'min', 'std', q25, q75, skew],
    'prob_2': ['mean', 'max', 'min', 'std', q25, q75, skew],
    'wealth_velocity': ['max', 'mean'],
    'education_num': ['max', 'mean'],
    'effort_yield': ['max', 'mean'],
    'age': ['min', 'max', 'mean']
}

# Apply aggregations to Train
train_meta = X_train_full.groupby('bag_id').agg({**agg_funcs, 'label': ['first']})
train_meta.columns = [f"{col[0]}_{col[1]}" if col[1] != 'first' else col[0] for col in train_meta.columns]
train_meta = train_meta.rename(columns={'label_<lambda_0>': 'label', 'label_first': 'label'})

X_meta = train_meta.drop('label', axis=1).fillna(0)
y_meta = train_meta['label'].astype(int)

# Apply aggregations to Test
test_meta = X_test_full.groupby('bag_id').agg(agg_funcs)
test_meta.columns = [f"{col[0]}_{col[1]}" for col in test_meta.columns]
test_meta = test_meta.fillna(0)

# Train the Meta Model
skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_meta_probs = np.zeros((len(X_meta), 3))
final_test_preds_prob = np.zeros((len(test_meta), 3))

for fold, (train_idx, val_idx) in enumerate(skf_meta.split(X_meta, y_meta)):
    X_tr, y_tr = X_meta.iloc[train_idx], y_meta.iloc[train_idx]
    X_va, y_va = X_meta.iloc[val_idx], y_meta.iloc[val_idx]

    # Meta model is heavily regularized to prevent overfitting on the synthetic probabilities
    model_l1 = lgb.LGBMClassifier(
        n_estimators=800, learning_rate=0.015, max_depth=4, num_leaves=15,
        class_weight='balanced', reg_lambda=3.0, reg_alpha=1.0, random_state=42, n_jobs=-1
    )
    model_l1.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(75, verbose=False)])

    oof_meta_probs[val_idx] = model_l1.predict_proba(X_va)
    final_test_preds_prob += model_l1.predict_proba(test_meta) / 5

# =====================================================================
# THRESHOLD OPTIMIZATION & FINAL OUTPUT
# =====================================================================
print("\n4. Optimizing Target Thresholds...")

def optimize_thresholds(y_true, y_probs):
    def f1_opt(weights):
        pred = np.argmax(y_probs * weights, axis=1)
        return -f1_score(y_true, pred, average='macro')
    # Use Nelder-Mead to solve the threshold optimization
    res = minimize(f1_opt, [1.0, 1.0, 1.0], method='Nelder-Mead')
    return res.x

best_weights = optimize_thresholds(y_meta, oof_meta_probs)
opt_preds_classes = np.argmax(oof_meta_probs * best_weights, axis=1)
final_f1 = f1_score(y_meta, opt_preds_classes, average='macro')

print("\n" + "="*50)
print(f"🏆 ULTIMATE STACKING MACRO F1 SCORE: {final_f1:.5f} 🏆")
print(f"Optimal Threshold Modifiers Applied: {np.round(best_weights, 3)}")
print("="*50)

# Generate final submission with exact same weights
final_predictions = np.argmax(final_test_preds_prob * best_weights, axis=1)

submission = pd.DataFrame({
    'bag_id': test_meta.index,
    'label': final_predictions
}).sort_values('bag_id').reset_index(drop=True)

submission.to_csv('submission_v4_ultimate.csv', index=False)
print("Saved final competitive predictions to 'submission_v4_ultimate.csv'")

1. Loading and Preprocessing Data...

2. STAGE 1: Training Instance Level Model (Weak Learners)...

3. STAGE 2: Aggregating Topology & Training Meta-Model...

4. Optimizing Target Thresholds...

🏆 ULTIMATE STACKING MACRO F1 SCORE: 0.74921 🏆
Optimal Threshold Modifiers Applied: [1.011 0.896 1.155]
Saved final competitive predictions to 'submission_v4_ultimate.csv'


In [ ]:
#colab
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize
import warnings

warnings.filterwarnings('ignore')

print("1. Loading and Preprocessing Data...")
train = pd.read_csv('data/raw/Coderush-26-ML-Train.csv')
test = pd.read_csv('data/raw/Coderush-26-ML-test.csv')

label_map = {'lower': 0, 'middle': 1, 'upper': 2}
train['label'] = train['label'].map(label_map)

train['is_train'] = 1
test['is_train'] = 0
test['label'] = -1
df = pd.concat([train, test], ignore_index=True)

# Feature Engineering
df['age'] = 1994 - df['year_of_birth']
df['net_capital'] = df['capital_gain'] - df['capital_loss']
df['wealth_velocity'] = df['net_capital_asset'] / (df['age'] + 1)
df['financial_stress'] = df['poverty_line_usd'] / (df['hours_per_week'] + 1)
df['effort_yield'] = df['education_num'] * df['hours_per_week']
df['is_capital_active'] = ((df['capital_gain'] > 0) | (df['capital_loss'] > 0)).astype(int)

df['education_tier'] = df['education_tier'].map({'Primary': 0, 'Secondary': 1, 'Higher': 2})
edu_map = {'Preschool': 0, '1st-4th': 1, '5th-6th': 2, '7th-8th': 3, '9th': 4, '10th': 5, '11th': 6, '12th': 7, 'HS-grad': 8, 'Some-college': 9, 'Assoc-voc': 10, 'Assoc-acdm': 11, 'Bachelors': 12, 'Masters': 13, 'Prof-school': 14, 'Doctorate': 15}
df['education'] = df['education'].map(edu_map)
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})

for col in ['native_country', 'occupation']:
    freq = df[col].value_counts() / len(df)
    df[f'{col}_freq'] = df[col].map(freq)

df = pd.get_dummies(df, columns=['relationship', 'race', 'workclass', 'marital_status'], drop_first=True)
for col in df.columns:
    if df[col].dtype == 'bool': df[col] = df[col].astype(int)

X_train_full = df[df['is_train'] == 1].drop(['is_train', 'year_of_birth', 'interview_mode', 'currency_code'], axis=1).copy()
X_test_full = df[df['is_train'] == 0].drop(['is_train', 'label', 'year_of_birth', 'interview_mode', 'currency_code'], axis=1).copy()

print("2. STAGE 1: Instance Probabilities...")
features = [c for c in X_train_full.columns if c not in ['label', 'bag_id', 'native_country', 'occupation']]
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_instance_probs = np.zeros((len(X_train_full), 3))
test_instance_probs = np.zeros((len(X_test_full), 3))

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train_full[features], X_train_full['label'], groups=X_train_full['bag_id'])):
    X_tr, y_tr = X_train_full[features].iloc[train_idx], X_train_full['label'].iloc[train_idx]
    X_va, y_va = X_train_full[features].iloc[val_idx], X_train_full['label'].iloc[val_idx]
    model_l0 = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.03, max_depth=7, num_leaves=63, class_weight='balanced', random_state=42, verbose=-1)
    model_l0.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_instance_probs[val_idx] = model_l0.predict_proba(X_va)
    test_instance_probs += model_l0.predict_proba(X_test_full[features]) / 5

for i in range(3):
    X_train_full[f'prob_{i}'] = oof_instance_probs[:, i]
    X_test_full[f'prob_{i}'] = test_instance_probs[:, i]

print("3. STAGE 2: Meta-Aggregation...")
def q25(x): return x.quantile(0.25)
def q75(x): return x.quantile(0.75)
def skew(x): return x.skew() if len(x) > 2 else 0

agg_funcs = {f'prob_{i}': ['mean', 'max', 'min', 'std', q25, q75, skew] for i in range(3)}
agg_funcs.update({'wealth_velocity': ['max', 'mean'], 'education_num': ['max', 'mean'], 'effort_yield': ['max', 'mean'], 'age': ['min', 'max', 'mean']})

train_meta = X_train_full.groupby('bag_id').agg({**agg_funcs, 'label': 'first'})
train_meta.columns = [f"{col[0]}_{col[1]}" if col[0] != 'label' else col[0] for col in train_meta.columns]
X_meta = train_meta.drop('label', axis=1).fillna(0)
y_meta = train_meta['label'].astype(int)

test_meta = X_test_full.groupby('bag_id').agg(agg_funcs)
test_meta.columns = [f"{col[0]}_{col[1]}" for col in test_meta.columns]
test_meta = test_meta.fillna(0)

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_meta_probs = np.zeros((len(X_meta), 3))
final_test_preds_prob = np.zeros((len(test_meta), 3))

for tr_idx, va_idx in skf_meta.split(X_meta, y_meta):
    m = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.015, max_depth=4, num_leaves=15, class_weight='balanced', reg_lambda=3.0, reg_alpha=1.0, random_state=42, verbose=-1)
    m.fit(X_meta.iloc[tr_idx], y_meta.iloc[tr_idx], eval_set=[(X_meta.iloc[va_idx], y_meta.iloc[va_idx])], callbacks=[lgb.early_stopping(75, verbose=False)])
    oof_meta_probs[va_idx] = m.predict_proba(X_meta.iloc[va_idx])
    final_test_preds_prob += m.predict_proba(test_meta) / 5

def f1_opt(w): return -f1_score(y_meta, np.argmax(oof_meta_probs * w, axis=1), average='macro')
res = minimize(f1_opt, [1.0, 1.0, 1.0], method='Nelder-Mead')

print(f"\n══════════════════════════════════════════════════\n⌒ ULTIMATE STACKING MACRO F1: {-res.fun:.5f} ⌒\n══════════════════════════════════════════════════")
final_predictions = np.argmax(final_test_preds_prob * res.x, axis=1)
pd.DataFrame({'bag_id': test_meta.index, 'label': final_predictions}).sort_values('bag_id').to_csv('submission_final_best.csv', index=False)
print("Saved final record-breaking predictions to 'submission_final_best.csv'")

1. Loading and Preprocessing Data...
2. STAGE 1: Instance Probabilities...
3. STAGE 2: Meta-Aggregation...

══════════════════════════════════════════════════
⌒ ULTIMATE STACKING MACRO F1: 0.76006 ⌒
══════════════════════════════════════════════════
Saved final record-breaking predictions to 'submission_final_best.csv'


In [ ]:
!pip install catboost -q

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize
import warnings

warnings.filterwarnings('ignore')

print("1. Loading and Preprocessing Data...")
train = pd.read_csv('data/raw/Coderush-26-ML-Train.csv')
test = pd.read_csv('data/raw/Coderush-26-ML-test.csv')

label_map = {'lower': 0, 'middle': 1, 'upper': 2}
train['label'] = train['label'].map(label_map)

train['is_train'] = 1
test['is_train'] = 0
test['label'] = -1
df = pd.concat([train, test], ignore_index=True)

# Advanced Instance Features
df['age'] = 1994 - df['year_of_birth']
df['net_capital'] = df['capital_gain'] - df['capital_loss']
df['wealth_velocity'] = df['net_capital_asset'] / (df['age'] + 1)
df['financial_stress'] = df['poverty_line_usd'] / (df['hours_per_week'] + 1)
df['effort_yield'] = df['education_num'] * df['hours_per_week']
df['capital_per_hour'] = df['net_capital_asset'] / (df['annual_hours_est'] + 1)
df['is_capital_active'] = ((df['capital_gain'] > 0) | (df['capital_loss'] > 0)).astype(int)

df['education_tier'] = df['education_tier'].map({'Primary': 0, 'Secondary': 1, 'Higher': 2}).fillna(-1)
edu_map = {'Preschool': 0, '1st-4th': 1, '5th-6th': 2, '7th-8th': 3, '9th': 4, '10th': 5, '11th': 6, '12th': 7, 'HS-grad': 8, 'Some-college': 9, 'Assoc-voc': 10, 'Assoc-acdm': 11, 'Bachelors': 12, 'Masters': 13, 'Prof-school': 14, 'Doctorate': 15}
df['education'] = df['education'].map(edu_map).fillna(-1)
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0}).fillna(-1)

for col in ['native_country', 'occupation']:
    freq = df[col].value_counts() / len(df)
    df[f'{col}_freq'] = df[col].map(freq)

df = pd.get_dummies(df.drop(columns=['interview_mode', 'currency_code', 'year_of_birth', 'native_country', 'occupation']), columns=['relationship', 'race', 'workclass', 'marital_status'], drop_first=True)
for col in df.columns:
    if df[col].dtype == 'bool': df[col] = df[col].astype(int)

X_train_full = df[df['is_train'] == 1].drop('is_train', axis=1).copy()
X_test_full = df[df['is_train'] == 0].drop(['is_train', 'label'], axis=1).copy()

print("2. STAGE 1: Instance Ensemble...")
features = [c for c in X_train_full.columns if c not in ['label', 'bag_id']]
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_instance_probs = np.zeros((len(X_train_full), 3))
test_instance_probs = np.zeros((len(X_test_full), 3))

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train_full[features], X_train_full['label'], groups=X_train_full['bag_id'])):
    X_tr, y_tr = X_train_full[features].iloc[train_idx], X_train_full['label'].iloc[train_idx]
    X_va, y_va = X_train_full[features].iloc[val_idx], X_train_full['label'].iloc[val_idx]
    m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.03, max_depth=7, num_leaves=63, class_weight='balanced', random_state=42, verbose=-1)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_instance_probs[val_idx] = m.predict_proba(X_va)
    test_instance_probs += m.predict_proba(X_test_full[features]) / 5

for i in range(3):
    X_train_full[f'p{i}'] = oof_instance_probs[:, i]
    X_test_full[f'p{i}'] = test_instance_probs[:, i]

print("3. STAGE 2: Meta-Ensemble Training...")
def q25(x): return x.quantile(0.25)
def q75(x): return x.quantile(0.75)
def skew(x): return x.skew() if len(x) > 2 else 0

agg_funcs = {f'p{i}': ['mean', 'max', 'min', 'std', q25, q75, skew] for i in range(3)}
agg_funcs.update({'wealth_velocity': ['max', 'mean', 'std'], 'education_num': ['max', 'mean'], 'effort_yield': ['max', 'mean'], 'age': ['min', 'max', 'mean']})

train_meta = X_train_full.groupby('bag_id').agg({**agg_funcs, 'label': 'first'})
train_meta.columns = [f"{c[0]}_{c[1]}" if not callable(c[1]) else f"{c[0]}_{c[1].__name__}" for c in train_meta.columns]
X_meta = train_meta.drop('label_first', axis=1).fillna(0)
y_meta = train_meta['label_first'].astype(int)

test_meta = X_test_full.groupby('bag_id').agg(agg_funcs)
test_meta.columns = [f"{c[0]}_{c[1]}" if not callable(c[1]) else f"{c[0]}_{c[1].__name__}" for c in test_meta.columns]

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_meta_probs = np.zeros((len(X_meta), 3))
final_test_preds_prob = np.zeros((len(test_meta), 3))

for tr_idx, va_idx in skf_meta.split(X_meta, y_meta):
    X_tr, y_tr, X_va, y_va = X_meta.iloc[tr_idx], y_meta.iloc[tr_idx], X_meta.iloc[va_idx], y_meta.iloc[va_idx]
    m1 = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.015, max_depth=4, class_weight='balanced', reg_lambda=3.0, random_state=42, verbose=-1)
    m2 = CatBoostClassifier(iterations=800, learning_rate=0.015, depth=4, auto_class_weights='Balanced', verbose=False, random_seed=42)
    m1.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(75, verbose=False)])
    m2.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_meta_probs[va_idx] = (m1.predict_proba(X_va) + m2.predict_proba(X_va)) / 2
    final_test_preds_prob += ((m1.predict_proba(test_meta) + m2.predict_proba(test_meta)) / 2) / 5

def f1_opt(w): return -f1_score(y_meta, np.argmax(oof_meta_probs * w, axis=1), average='macro')
res = minimize(f1_opt, [1.0, 1.0, 1.0], method='Nelder-Mead')

print(f"\n══════════════════════════════════════════════════\n⌒ FINAL GRANDMASTER MACRO F1: {-res.fun:.5f} ⌒\n══════════════════════════════════════════════════")
final_predictions = np.argmax(final_test_preds_prob * res.x, axis=1)
pd.DataFrame({'bag_id': test_meta.index, 'label': final_predictions}).to_csv('submission_v7_final.csv', index=False)
print("Saved final predictions to 'submission_v7_final.csv'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00
1. Loading and Preprocessing Data...
2. STAGE 1: Instance Ensemble...
3. STAGE 2: Meta-Ensemble Training...

══════════════════════════════════════════════════
⌒ FINAL GRANDMASTER MACRO F1: 0.75672 ⌒
══════════════════════════════════════════════════
Saved final predictions to 'submission_v7_final.csv'


In [ ]:
#deepseek:-
# ============================================
# Economic Class Classification – Two‑Stage Stacking (Legal Version)
# ============================================
# No test set leakage – all statistics computed from training only.
# Saves 'submission.csv' and 'best_model.joblib'.
# ============================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ------------------------------
# 1. LOAD DATA
# ------------------------------
train = pd.read_csv('data/raw/Coderush-26-ML-Train.csv')
test  = pd.read_csv('data/raw/Coderush-26-ML-test.csv')
print(f"Train shape: {train.shape}, Test shape: {test.shape}")

# Map labels to numeric
label_map = {'lower': 0, 'middle': 1, 'upper': 2}
train['label'] = train['label'].map(label_map)

# ------------------------------
# 2. FEATURE ENGINEERING (TRAIN ONLY – NO LEAKAGE)
# ------------------------------
def add_features(df, is_train=True):
    """Add derived features to a dataframe (train or test)."""
    df = df.copy()
    df['age'] = 1994 - df['year_of_birth']
    df['net_capital'] = df['capital_gain'] - df['capital_loss']
    df['wealth_velocity'] = df['net_capital_asset'] / (df['age'] + 1)
    df['financial_stress'] = df['poverty_line_usd'] / (df['hours_per_week'] + 1)
    df['effort_yield'] = df['education_num'] * df['hours_per_week']
    df['is_capital_active'] = ((df['capital_gain'] > 0) | (df['capital_loss'] > 0)).astype(int)

    # Ordinal mappings
    df['education_tier'] = df['education_tier'].map({'Primary': 0, 'Secondary': 1, 'Higher': 2})
    edu_map = {
        'Preschool': 0, '1st-4th': 1, '5th-6th': 2, '7th-8th': 3, '9th': 4, '10th': 5,
        '11th': 6, '12th': 7, 'HS-grad': 8, 'Some-college': 9, 'Assoc-voc': 10,
        'Assoc-acdm': 11, 'Bachelors': 12, 'Masters': 13, 'Prof-school': 14, 'Doctorate': 15
    }
    df['education'] = df['education'].map(edu_map)
    df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})

    # Drop original year_of_birth (replaced by age)
    df = df.drop(columns=['year_of_birth'], errors='ignore')
    return df

# Apply to train and test separately (no mixing)
train = add_features(train, is_train=True)
test = add_features(test, is_train=False)

# ------------------------------
# 3. FREQUENCY ENCODING (TRAIN ONLY, THEN MAP TO TEST)
# ------------------------------
for col in ['native_country', 'occupation']:
    # Compute frequencies from TRAINING only
    freq = train[col].value_counts(normalize=True)
    train[f'{col}_freq'] = train[col].map(freq)
    test[f'{col}_freq'] = test[col].map(freq).fillna(0)   # unseen categories -> 0
    # Drop original high‑cardinality columns
    train = train.drop(columns=[col])
    test = test.drop(columns=[col])

# ------------------------------
# 4. ONE‑HOT ENCODING (TRAIN ONLY, THEN ALIGN TEST)
# ------------------------------
low_card = ['relationship', 'race', 'workclass', 'marital_status']
train = pd.get_dummies(train, columns=low_card, drop_first=True)
test = pd.get_dummies(test, columns=low_card, drop_first=True)

# Align test columns with train
missing_cols = set(train.columns) - set(test.columns)
for col in missing_cols:
    test[col] = 0
test = test[train.columns]   # same order

# Drop constant/id columns that remain
drop_cols = ['interview_mode', 'currency_code', 'poverty_line_usd',
             'survey_year', 'processing_flag', 'person_idx', 'interviewer_id']
train = train.drop(columns=[c for c in drop_cols if c in train.columns], errors='ignore')
test = test.drop(columns=[c for c in drop_cols if c in test.columns], errors='ignore')

# Ensure label is not used as feature (training only)
X_train_full = train.drop(columns=['label'], errors='ignore')
y_train_full = train['label']
X_test_full = test.drop(columns=['label'], errors='ignore')

# ------------------------------
# 5. STAGE 1: INSTANCE‑LEVEL MODEL (LightGBM)
# ------------------------------
print("\n=== Stage 1: Instance‑level LightGBM ===")
features = [c for c in X_train_full.columns if c not in ['bag_id']]
X = X_train_full[features]
y = y_train_full
groups = X_train_full['bag_id']

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_instance_probs = np.zeros((len(X_train_full), 3))
test_instance_probs = np.zeros((len(X_test_full), 3))

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    model_l0 = lgb.LGBMClassifier(
        n_estimators=1500, learning_rate=0.03, max_depth=7, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, class_weight='balanced',
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_l0.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                 callbacks=[lgb.early_stopping(50, verbose=False)])

    oof_instance_probs[val_idx] = model_l0.predict_proba(X_va)
    test_instance_probs += model_l0.predict_proba(X_test_full[features]) / 5

# Attach probabilities AND label back to X_train_full for grouping
for i in range(3):
    X_train_full[f'prob_{i}'] = oof_instance_probs[:, i]
    X_test_full[f'prob_{i}'] = test_instance_probs[:, i]
X_train_full['label'] = y_train_full.values

# ------------------------------
# 6. STAGE 2: BAG‑LEVEL META‑MODEL
# ------------------------------
print("\n=== Stage 2: Bag‑level meta‑model ===")
def q25(x): return x.quantile(0.25)
def q75(x): return x.quantile(0.75)
def skew(x): return x.skew() if len(x) > 2 else 0

agg_funcs = {
    'prob_0': ['mean', 'max', 'min', 'std', q25, q75, skew],
    'prob_1': ['mean', 'max', 'min', 'std', q25, q75, skew],
    'prob_2': ['mean', 'max', 'min', 'std', q25, q75, skew],
    'wealth_velocity': ['max', 'mean', 'std'],
    'education_num': ['max', 'mean'],
    'effort_yield': ['max', 'mean'],
    'age': ['min', 'max', 'mean']
}

# Train meta
train_meta = X_train_full.groupby('bag_id').agg({**agg_funcs, 'label': 'first'})
train_meta.columns = [f"{col[0]}_{col[1]}" if col[0] != 'label' else col[0] for col in train_meta.columns]
X_meta = train_meta.drop('label', axis=1).fillna(0)
y_meta = train_meta['label'].astype(int)

# Test meta
test_meta = X_test_full.groupby('bag_id').agg(agg_funcs)
test_meta.columns = [f"{col[0]}_{col[1]}" for col in test_meta.columns]
test_meta = test_meta.fillna(0)

# Cross‑validation for meta‑model
skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_meta_probs = np.zeros((len(X_meta), 3))
final_test_probs = np.zeros((len(test_meta), 3))

for fold, (tr_idx, va_idx) in enumerate(skf_meta.split(X_meta, y_meta)):
    X_tr, y_tr = X_meta.iloc[tr_idx], y_meta.iloc[tr_idx]
    X_va, y_va = X_meta.iloc[va_idx], y_meta.iloc[va_idx]

    model_l1 = lgb.LGBMClassifier(
        n_estimators=800, learning_rate=0.015, max_depth=4, num_leaves=15,
        class_weight='balanced', reg_lambda=3.0, reg_alpha=1.0,
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_l1.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                 callbacks=[lgb.early_stopping(75, verbose=False)])

    oof_meta_probs[va_idx] = model_l1.predict_proba(X_va)
    final_test_probs += model_l1.predict_proba(test_meta) / 5

# ------------------------------
# 7. THRESHOLD OPTIMIZATION (for macro F1)
# ------------------------------
print("\n=== Threshold optimization ===")
def optimize_thresholds(y_true, y_probs):
    def f1_opt(weights):
        pred = np.argmax(y_probs * weights, axis=1)
        return -f1_score(y_true, pred, average='macro')
    res = minimize(f1_opt, [1.0, 1.0, 1.0], method='Nelder-Mead')
    return res.x

best_weights = optimize_thresholds(y_meta, oof_meta_probs)
final_val_preds = np.argmax(oof_meta_probs * best_weights, axis=1)
final_f1 = f1_score(y_meta, final_val_preds, average='macro')

print(f"\n{'='*50}")
print(f"⚙ Final validation Macro F1: {final_f1:.5f} ⚙")
print(f"Optimal weights: {best_weights.round(3)}")
print(f"{'='*50}")

# ------------------------------
# 8. FINAL PREDICTION ON TEST SET
# ------------------------------
test_preds = np.argmax(final_test_probs * best_weights, axis=1)
inv_label_map = {0: 'lower', 1: 'middle', 2: 'upper'}
submission = pd.DataFrame({
    'bag_id': test_meta.index,
    'label': [inv_label_map[p] for p in test_preds]
}).sort_values('bag_id').reset_index(drop=True)

submission.to_csv('Deep_submission.csv', index=False)
print("\n✅ Submission saved as 'Deep_submission.csv'")
joblib.dump(model_l1, 'best_model.joblib')
print("✅ Model saved as 'best_model.joblib'")

Train shape: (16776, 29), Test shape: (1981, 28)

=== Stage 1: Instance‑level LightGBM ===

=== Stage 2: Bag‑level meta‑model ===

=== Threshold optimization ===

⚙ Final validation Macro F1: 0.74092 ⚙
Optimal weights: [1.056 0.974 1.009]

✅ Submission saved as 'Deep_submission.csv'
✅ Model saved as 'best_model.joblib'
